In [1]:
# CELL 1: IMPORT
import sys
sys.path.insert(0, '/home/duoc/workflow/api')

import asyncio
from io import BytesIO
from PIL import Image
import numpy as np

from app.rag.services.config import FINAL_K, RETRIEVE_K, PREFILTER_K
from app.rag.services.keyword import KeywordRAG
from app.rag.services.comment import CommentRAG
from app.rag.services.social import SocialPostRAG
from app.rag.services.image_rag import ImageRAG
from app.rag.services.embedder import get_embedder
from app.rag.services.reranker import get_reranker

print('✅ Import OK')

✅ Import OK


In [2]:
# CELL 2: INIT RAG instances
krag = KeywordRAG()
crag = CommentRAG()
srag = SocialPostRAG()
erag = get_embedder()
irag = ImageRAG(erag)

BUSINESS_ID = "test_biz_001"
SOURCE_ID = "test_src_001"

print('✅ RAG instances OK')
print(f'   RETRIEVE_K={RETRIEVE_K}, PREFILTER_K={PREFILTER_K}, FINAL_K={FINAL_K}')

✅ RAG instances OK
   RETRIEVE_K=40, PREFILTER_K=15, FINAL_K=5


In [3]:
# CELL 3: TEST KEYWORD + COMMENT ADD
async def test_add_keyword_comment():
    # Keyword
    kw_texts = [
        "Python web development",
        "Machine learning NLP",
        "Cloud computing AWS"
    ]
    for i, kw in enumerate(kw_texts):
        result = await krag.add(kw, business_id=BUSINESS_ID, source_id=f"kw_{i}")
        print(f"  Keyword {i}: {result} - '{kw}'")
    
    # Comment
    comments = [
        "Great tutorial on FastAPI, very clear and practical examples!",
        "How to optimize database queries for better performance?",
        "Love the new React hooks, makes state management much easier."
    ]
    for i, cmt in enumerate(comments):
        result = await crag.add(cmt, business_id=BUSINESS_ID, source_id=f"cmt_{i}")
        print(f"  Comment {i}: {result}")

await test_add_keyword_comment()
print('✅ Keyword + Comment ADD OK')

  Keyword 0: ok - 'Python web development'
  Keyword 1: ok - 'Machine learning NLP'
  Keyword 2: ok - 'Cloud computing AWS'
  Comment 0: ok
  Comment 1: ok
  Comment 2: ok
✅ Keyword + Comment ADD OK


In [4]:
# CELL 4: TEST SOCIAL + DUPLICATE CHECK
async def test_social_and_duplicate():
    social_items = [
        {
            "title": "Top 10 Python Tips",
            "body": "Tip 1: Use list comprehension\n\nTip 2: Leverage built-in functions\n\nTip 3: Profile before optimize",
            "cta": "Read full article here →"
        },
        {
            "title": "React Performance Guide",
            "body": "Avoid inline functions\n\nUse React.memo for components\n\nOptimize re-renders",
            "cta": "Learn more"
        }
    ]
    
    for i, item in enumerate(social_items):
        result = await srag.add(
            business_id=BUSINESS_ID,
            source_id=f"social_{i}",
            title=item["title"],
            body=item["body"],
            cta=item["cta"]
        )
        print(f"  Social {i}: {result} - '{item['title']}'")
    
    # Test duplicate
    dup_result = await srag.add(
        business_id=BUSINESS_ID,
        source_id="social_0",
        title="Same title",
        body="Different body",
        cta="CTA"
    )
    print(f"  Duplicate check: {dup_result} (should be 'duplicate')")

await test_social_and_duplicate()
print('✅ Social + Duplicate OK')

  Social 0: ok - 'Top 10 Python Tips'
  Social 1: ok - 'React Performance Guide'
  Duplicate check: duplicate (should be 'duplicate')
✅ Social + Duplicate OK


In [5]:
# CELL 5: TEST SEARCH ALL TYPES
async def test_search():
    queries = [
        ("python development", krag, "keyword"),
        ("database optimization", crag, "comment"),
        ("performance react", srag, "social"),
    ]
    
    for query, rag, rag_type in queries:
        results = await rag.search(query, business_id=BUSINESS_ID, top_k=3)
        print(f"\n🔍 {rag_type.upper()}: '{query}'")
        if results:
            for i, chunk in enumerate(results, 1):
                print(f"  {i}. Score: {chunk.score:.3f} | {chunk.text[:60]}...")
        else:
            print(f"  ❌ No results")

await test_search()
print('\n✅ Search ALL TYPES OK')

`torch_dtype` is deprecated! Use `dtype` instead!



🔍 KEYWORD: 'python development'
  1. Score: 0.991 | Python web development...
  2. Score: 0.002 | Machine learning NLP...
  3. Score: 0.000 | Cloud computing AWS...

🔍 COMMENT: 'database optimization'
  1. Score: 0.996 | How to optimize database queries for better performance?...
  2. Score: 0.000 | Great tutorial on FastAPI, very clear and practical examples...
  3. Score: 0.000 | Love the new React hooks, makes state management much easier...

🔍 SOCIAL: 'performance react'
  1. Score: 0.988 | React Performance Guide...
  2. Score: 0.017 | Optimize re-renders...
  3. Score: 0.001 | Use React.memo for components...

✅ Search ALL TYPES OK


In [6]:
# CELL 6: TEST STATS + CONCURRENT ADD
async def test_stats_and_concurrent():
    # Stats before
    print("📊 Stats BEFORE:")
    for name, rag in [("keyword", krag), ("comment", crag), ("social", srag)]:
        stats = rag.stats()
        print(f"  {name}: {stats}")
    
    # Concurrent add
    print("\n⚡ Concurrent add 5 keywords...")
    tasks = []
    for i in range(5):
        task = krag.add(
            f"Concurrent keyword {i}",
            business_id=BUSINESS_ID,
            source_id=f"concurrent_kw_{i}"
        )
        tasks.append(task)
    
    results = await asyncio.gather(*tasks)
    print(f"  Results: {results}")
    
    # Stats after
    print("\n📊 Stats AFTER:")
    for name, rag in [("keyword", krag), ("comment", crag), ("social", srag)]:
        stats = rag.stats()
        print(f"  {name}: {stats}")

await test_stats_and_concurrent()
print('\n✅ Stats + Concurrent OK')

📊 Stats BEFORE:
  keyword: {'chunks': 3, 'faiss_vectors': 3, 'documents': 3}
  comment: {'chunks': 3, 'faiss_vectors': 3, 'documents': 3}
  social: {'chunks': 10, 'faiss_vectors': 10, 'documents': 2}

⚡ Concurrent add 5 keywords...
  Results: ['ok', 'ok', 'ok', 'ok', 'ok']

📊 Stats AFTER:
  keyword: {'chunks': 8, 'faiss_vectors': 8, 'documents': 8}
  comment: {'chunks': 3, 'faiss_vectors': 3, 'documents': 3}
  social: {'chunks': 10, 'faiss_vectors': 10, 'documents': 2}

✅ Stats + Concurrent OK


In [7]:
# CELL 7: TEST IMAGE RAG (nếu có GPU/model)
async def test_image_rag():
    print("🖼️  Testing ImageRAG...")
    
    # Tạo test image (solid color)
    test_img = Image.new('RGB', (100, 100), color='red')
    
    try:
        # Single add
        result = await irag.add(test_img, "test_img_001", category="test")
        print(f"  Single add: {result}")
        
        # Search
        search_results = await irag.search_by_text("red color", k=3)
        print(f"  Search results: {len(search_results)} found")
        if search_results:
            for r in search_results:
                print(f"    - {r.get('image_id')}: score={r.get('score'):.3f}")
        
        # Stats
        stats = irag.stats()
        print(f"  Stats: {stats}")
    except Exception as e:
        print(f"  ⚠️  ImageRAG skipped: {type(e).__name__} (captioner may need GPU)")

try:
    await test_image_rag()
    print('✅ ImageRAG test OK')
except Exception as e:
    print(f'⚠️  ImageRAG test error: {e}')

🖼️  Testing ImageRAG...


The following generation flags are not valid and may be ignored: ['early_stopping']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  Single add: ok
  Search results: 1 found
    - test_img_001: score=0.605
  Stats: {'images': 1, 'faiss_vectors': 1}
✅ ImageRAG test OK


In [8]:
# CELL 8: EDGE CASES
async def test_edge_cases():
    print("🧪 Testing edge cases...")
    
    # Empty input
    r1 = await krag.add("", business_id=BUSINESS_ID, source_id="empty")
    print(f"  Empty keyword: {r1} (expected 'skipped')")
    
    # Whitespace only
    r2 = await crag.add("   \n\n  ", business_id=BUSINESS_ID, source_id="ws")
    print(f"  Whitespace comment: {r2} (expected 'skipped')")
    
    # Empty query search
    r3 = await krag.search("", business_id=BUSINESS_ID)
    print(f"  Empty query search: {len(r3)} results (expected 0)")
    
    # Search unknown business_id
    r4 = await krag.search("python", business_id="unknown_biz")
    print(f"  Unknown business_id search: {len(r4)} results (expected 0)")
    
    print('✅ Edge cases OK')

await test_edge_cases()

🧪 Testing edge cases...
  Empty keyword: skipped (expected 'skipped')
  Whitespace comment: skipped (expected 'skipped')
  Empty query search: 0 results (expected 0)
  Unknown business_id search: 0 results (expected 0)
✅ Edge cases OK
